# MINI Cells — Experiment 010: Residual Adaptive Halting Probe

This experiment does **not retrain** the models. It loads the 2M-token Experiment 009 checkpoints and asks whether the already-learned 1D and K=4 2D dynamics expose a usable stopping signal. For causal autoregressive evaluation, each sample is one 128-token prefix (batch size 1), the stopping rule observes only the final-prefix readout residual, and only the next token after that prefix is scored.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path('/kaggle/working/mini-cells')
REF = os.environ.get('MINICELLS_REF', 'codex/experiment-010-adaptive-halting')
os.chdir('/kaggle/working')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REF, 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
print('ref:', REF)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
if not torch.cuda.is_available():
    raise RuntimeError('Experiment 010 requires CUDA')


## Load Experiment 009 artifacts

If this notebook is not running in the same Kaggle session as Experiment 009, publish Experiment 009 first. The cell below imports its curated result artifacts from `kaggle/experiment-009-results`.


In [ ]:
LOCAL_009 = ROOT / 'results' / 'language-2d-latent-tissue-v1'
ARTIFACT_009 = ROOT / 'artifacts' / 'experiments' / '009-2d-latent-tissue'
if not ((LOCAL_009 / 'minicells-v2-2m.pt').is_file() and (LOCAL_009 / 'minicells-2d-k4-2m.pt').is_file()):
    fetch = subprocess.run(['git', 'fetch', 'origin', 'kaggle/experiment-009-results'], cwd=ROOT, text=True, capture_output=True)
    if fetch.returncode != 0:
        print(fetch.stderr)
        raise RuntimeError('Experiment 009 results branch is unavailable. Publish Experiment 009 first.')
    subprocess.run(['git', 'checkout', 'FETCH_HEAD', '--', 'artifacts/experiments/009-2d-latent-tissue'], cwd=ROOT, check=True)
print('009 source ready:', LOCAL_009 if LOCAL_009.exists() else ARTIFACT_009)


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_language_halting.py', 'tests/test_language_2d.py', '-q'], cwd=ROOT, check=True)


In [ ]:
subprocess.run([sys.executable, 'scripts/run_language_halting_probe.py'], cwd=ROOT, check=True)


In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

OUT = ROOT / 'results' / 'language-adaptive-halting-v1'
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
sweep = pd.read_csv(OUT / 'halting-sweep.csv')
display(Markdown(f"## {decision['status']}: {decision['diagnosis']}"))
display(sweep)
display(Markdown('### Best safe halting points'))
print(json.dumps(decision['best'], indent=2))
for name in ['ppl-vs-iterations.png', 'walltime-vs-iterations.png']:
    display(Image(filename=str(OUT / name)))


In [ ]:
# Set this to True after reviewing decision.json and the two trade-off plots.
PUBLISH = False
if PUBLISH:
    subprocess.run([sys.executable, 'scripts/publish_experiment_010_results.py', '--push'], cwd=ROOT, check=True)
